# About this model

 BERT (Bidirectional Encoder Representations from Transformers) is a powerful language model developed by Google. Think of it as a model that "reads" and "understands" text by looking at words in context.

 NOTE: This model takes lot of time but efficient
 

# Deep Learning with BERT

In this notebook, we'll train a **DistilBERT** model for ticket classification. Deep learning models like BERT are state-of-the-art for NLP tasks because they understand the *context* of words, not just their frequency (like TF-IDF).

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch transformers tqdm

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizer
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import sys
import os

# Add project root to path for src imports
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_loader import load_data, TicketDataset
from src.bert_model import TicketBERT

c:\Users\dharn\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Preparation
Let's look at the data we are feeding into the model.

In [ ]:
# Load raw data
df = load_data('data/tickets.csv')

# Show first 5 rows to understand the input
print("Sample Data:")
display(df.head())

✓ Loaded 8469 tickets
✓ Columns: ['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'category', 'subject', 'description', 'Ticket Status', 'Resolution', 'priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']
✓ Shape: (8469, 17)
Sample Data:


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,category,subject,description,Ticket Status,Resolution,priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [ ]:
# Encode Text Categories to Integers
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['category'])
n_classes = len(list(le.classes_))
print(f"Classes detected ({n_classes}): {le.classes_}")

Classes detected (5): ['Billing inquiry' 'Cancellation request' 'Product inquiry'
 'Refund request' 'Technical issue']


In [ ]:
# Split into Train and Validation sets
X_train, X_test, y_train, y_test = train_test_split(
    df['description'], 
    df['label_encoded'], 
    test_size=0.2, 
    random_state=42
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Train size: 6775 | Test size: 1694


## 2. Tokenization & Datasets
Initialize the BERT tokenizer and create PyTorch Datasets.

In [ ]:
# Initialize Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
# Configuration
MAX_LEN = 128
BATCH_SIZE = 16

# Create Datasets
train_dataset = TicketDataset(
    texts=X_train.to_numpy(),
    labels=y_train.to_numpy(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TicketDataset(
    texts=X_test.to_numpy(),
    labels=y_test.to_numpy(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

## 3. Model Setup
Initialize the model, optimizer, and training reference (GPU/CPU).

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Training on: {device}")

Training on: cpu


In [ ]:
model = TicketBERT(n_classes=n_classes)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 362.03it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Training Loop

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("-" * 10)
    
    # Training Step
    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        _, preds = torch.max(outputs, dim=1)
        loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct_predictions += torch.sum(preds == targets)
        total_predictions += targets.shape[0]
    
    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions.double() / total_predictions
    print(f"Train Loss: {avg_loss:.4f} | Train Acc: {accuracy:.4f}")

Epoch 1/3
----------


Training: 100%|██████████| 424/424 [25:16<00:00,  3.58s/it]


Train Loss: 1.6176 | Train Acc: 0.1979
Epoch 2/3
----------


Training:  67%|██████▋   | 286/424 [11:48<05:41,  2.48s/it]


KeyboardInterrupt: 

## 5. Evaluation

In [ ]:
model.eval()
correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == targets)
        total_predictions += targets.shape[0]

val_acc = correct_predictions.double() / total_predictions
print(f"Validation Accuracy: {val_acc:.4f}")

## 6. Real-time Inference (See it work!)
Let's test the model on some random examples to see how it performs on actual text.

In [ ]:
def predict_custom(text, model, tokenizer, device, label_encoder):
    model.eval()
    encoding = tokenizer.encode_plus(
        text,
        max_length=128,
        add_special_tokens=True,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device).unsqueeze(0)  # Add batch dim
    attention_mask = encoding['attention_mask'].to(device).unsqueeze(0)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        _, preds = torch.max(outputs, dim=1)
        
    return label_encoder.inverse_transform(preds.cpu().numpy())[0]

# Test on 5 random validation samples
print("\n--- Inference Results ---")
indices = np.random.choice(len(X_test), 5)
for i in indices:
    text = X_test.iloc[i]
    true_label = le.inverse_transform([y_test.iloc[i]])[0]
    pred_label = predict_custom(text, model, tokenizer, device, le)
    
    print(f"\nTicket: '{text[:80]}...'")
    print(f"Actual: {true_label} | Predicted: {pred_label}")
    if true_label == pred_label:
        print("✅ Correct")
    else:
        print("❌ Incorrect")

In [ ]:
# Save the model state
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/bert_classifier.pth')
print("\nModel saved to models/bert_classifier.pth")